# Stage 0 — Simple Enterprise Policy Agent (Colab setup + run)

This notebook does three things: installs Ollama and the Python
dependencies, pulls the local LLM + embedding models, and then runs the
existing project code (`policy-agent-stage0.zip` — the same code already
delivered to you) — ingestion, the plan's smoke test, and an interactive
chat, all right here.

It does **not** re-embed the source code as notebook cells — the project
stays as real `.py` files, uploaded once at the start. Every setup step
below is plain Python (the `subprocess` module), not shell one-liners.

**Before running:** have `policy-agent-stage0.zip` ready on your computer
(the one already sent to you).

**Optional but recommended:** `Runtime → Change runtime type → T4 GPU` —
Ollama uses it automatically if present.

**Note:** Colab runtimes are ephemeral. Everything here (models,
`qdrant_data/`) disappears when the runtime recycles — re-running the
notebook rebuilds it in a couple of minutes. See the optional "persist to
Drive" cell at the end if you want it to survive across sessions.


## 1. Upload the project
Run this cell, then pick `policy-agent-stage0.zip` from your computer when the file picker appears.

In [ ]:
from google.colab import files

uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))


Saving policy-agent-stage0.zip to policy-agent-stage0.zip
Uploaded: ['policy-agent-stage0.zip']


In [ ]:
import io
import os
import zipfile

zip_name = next(iter(uploaded.keys()))
with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as zf:
    zf.extractall("/content")

os.chdir("/content/policy-agent")
print("Working directory:", os.getcwd())
print()
for root, dirs, filenames in os.walk("."):
    dirs[:] = [d for d in dirs if d not in (".git", "__pycache__")]
    for name in filenames:
        print(os.path.relpath(os.path.join(root, name)))


Working directory: /content/policy-agent

.env.example
requirements.txt
README.md
scripts/smoke_test.py
scripts/generate_sample_data.py
app/__init__.py
app/streamlit_app.py
app/db.py
app/agent.py
app/config.py
app/ingest.py
app/tools.py
app/main.py
data/employees.csv
data/policies/code_of_conduct.pdf
data/policies/security_policy.pdf
data/policies/remote_work_policy.pdf
data/policies/pto_policy.pdf
data/policies/expense_policy.pdf


## 2. (Optional) Check for a GPU
Ollama will use it automatically if available. It's fine if this shows no GPU.

## 3. Install Ollama and start the server

Plain Python (`subprocess`) downloading and running Ollama's install
script, then starting `ollama serve` in the background.


In [ ]:
import subprocess

# Ollama's installer needs zstd to unpack its release archive — Colab's
# base image doesn't have it, which is what caused the
# "This version requires zstd for extraction" failure. Install it first.
apt_update = subprocess.run(["apt-get", "update", "-qq"], capture_output=True)
apt_install = subprocess.run(["apt-get", "install", "-y", "-qq", "zstd"], capture_output=True)
if apt_install.returncode != 0:
    print(apt_install.stdout.decode(errors="replace"))
    print(apt_install.stderr.decode(errors="replace"))
    raise RuntimeError(f"Installing zstd failed (exit code {apt_install.returncode}) — see output above.")
print("zstd installed.")

# Equivalent to `curl -fsSL https://ollama.com/install.sh | sh`, written as
# Python instead of a shell one-liner. Run with bash rather than /bin/sh:
# on Colab's Debian base, /bin/sh is dash, and Ollama's installer uses a
# couple of bash-only constructs that dash rejects.
download = subprocess.run(
    ["curl", "-fsSL", "https://ollama.com/install.sh"],
    capture_output=True, check=True,
)
install = subprocess.run(["bash"], input=download.stdout, capture_output=True)
print(install.stdout.decode(errors="replace"))
if install.returncode != 0:
    print(install.stderr.decode(errors="replace"))
    raise RuntimeError(f"Ollama install script failed (exit code {install.returncode}) — see output above.")
print("Ollama installed.")


zstd installed.

Ollama installed.


In [ ]:
import subprocess, time, requests

ollama_log = open("/content/ollama_serve.log", "w")
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=subprocess.STDOUT)

for attempt in range(30):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        if r.status_code == 200:
            print("Ollama server is up.")
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama server did not start — check /content/ollama_serve.log")


Ollama server is up.


## 4. Pull the models

- `qwen2.5:1.5b` — small, fast, supports tool-calling.
- `nomic-embed-text` — embedding model.

If your runtime has a GPU and you want better answers, swap
`qwen2.5:1.5b` for e.g. `qwen2.5:7b` or `llama3.1:8b` here **and** set
`OLLAMA_CHAT_MODEL` in step 5 below to match.


In [ ]:
import subprocess

for model in ["qwen2.5:1.5b", "nomic-embed-text"]:
    print(f"Pulling {model} ...")
    subprocess.run(["ollama", "pull", model], check=True)
print("Models ready.")


Pulling qwen2.5:1.5b ...
Pulling nomic-embed-text ...
Models ready.


## 5. Install Python dependencies

In [ ]:
import subprocess, sys

packages = [
    "fastapi", "uvicorn[standard]", "streamlit", "requests", "python-dotenv",
    "langchain-core", "langchain-ollama>=0.1.3", "langchain-text-splitters",
    "qdrant-client>=1.10", "duckdb", "PyMuPDF>=1.24", "pydantic>=2.6",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print("Dependencies installed.")


Dependencies installed.


Optional: override any setting from `.env.example` before the app is
imported (defaults are fine for Stage 0 — skip this cell if you don't need
to change anything).


In [ ]:
import os

# Example — uncomment to use a bigger chat model:
# os.environ["OLLAMA_CHAT_MODEL"] = "qwen2.5:7b"


## 6. Ingest the documents into Qdrant

The sample policy PDFs and `employees.csv` are already included in the
uploaded project — this just parses, chunks, embeds, and loads them into a
local on-disk Qdrant collection (no Docker needed).


In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "app.ingest"], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'app.ingest'], returncode=0)

## 7. Run the plan's acceptance test

> "Can interns work remotely?" should hit `search_policies`;
> "Who manages Alice?" should hit `search_employee`.


In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "scripts/smoke_test.py"], check=True)


CompletedProcess(args=['/usr/bin/python3', 'scripts/smoke_test.py'], returncode=0)

## 8. Ask your own questions, right here in the notebook

Calls the agent directly — no FastAPI/Streamlit needed for this.


In [ ]:
import sys
sys.path.insert(0, "/content/policy-agent")
from app.agent import run_agent

def ask(question: str):
    result = run_agent(question)
    tools_used = [c["name"] for c in result["tool_calls"]]
    print(f"Q: {question}")
    print(f"Tools used: {tools_used}")
    print(f"A: {result['answer']}\n")
    for call in result["tool_calls"]:
        print(f"  [{call['name']}({call['args']}) ->]")
        print(f"  {call['result'][:300]}{'...' if len(call['result']) > 300 else ''}\n")
    return result

ask("Can interns work remotely?")
ask("Who manages Alice?")


Q: Can interns work remotely?
Tools used: ['search_policies']
A: Based on the Remote Work Policy (Effective Jan 2026), interns and part-time contractors are NOT eligible for remote work under this policy and are expected to work on-site for the duration of their placement. Therefore, no interns can currently work remotely at Acme Corp.

For more detailed information about remote work policies, please refer to the Remote Work Policy document (effective Jan 2026).

  [search_policies({'query': 'can interns work remotely'}) ->]
  [source: remote_work_policy.pdf, score: 0.630]
Acme Corp  Remote Work Policy (Effective Jan 2026)
1. Eligibility
--------------
Full-time employees who have completed their 90-day probationary period are eligible to work
remotely up to 3 days per week, subject to manager approval. Interns and part-t...

Q: Who manages Alice?
Tools used: ['search_employee']
A: Alice Nakamura is managed by David Reyes in the Engineering department at Acme Corp. Her email address is

{'answer': 'Alice Nakamura is managed by David Reyes in the Engineering department at Acme Corp. Her email address is alice.nakamura@acmecorp.example and she works from Magdeburg, Germany on a full-time basis.',
 'tool_calls': [{'name': 'search_employee',
   'args': {'name': 'Alice'},
   'result': 'Alice Nakamura — Senior Software Engineer, Engineering. Manager: David Reyes. Email: alice.nakamura@acmecorp.example. Location: Magdeburg, DE. Type: Full-time.'}]}

In [ ]:
# Try your own — some ideas:
ask("How many PTO days do I get per year?")
ask("What's the dog-walking policy?")  # not in the docs — should say it doesn't know, not make something up


Q: How many PTO days do I get per year?
Tools used: []
A: The number of PTO (Paid Time Off) days you receive each year can vary depending on your role and the specific policies of Acme Corp. Typically, this amount is outlined in your employment contract or within the company's employee handbook.

To find out exactly how many PTO days you are entitled to per year, I recommend checking either your employment contract or the employee handbook provided by Acme Corp. If these documents aren't available online, you may need to contact HR directly for clarification.

Q: What's the dog-walking policy?
Tools used: []
A: I'm sorry, but I don't have access to specific company policies or documents. Could you please provide more context about what kind of information you're looking for regarding the dog-walking policy at Acme Corp? This could help me find relevant information that might be available in our HR documentation.



{'answer': "I'm sorry, but I don't have access to specific company policies or documents. Could you please provide more context about what kind of information you're looking for regarding the dog-walking policy at Acme Corp? This could help me find relevant information that might be available in our HR documentation.",
 'tool_calls': []}

## 9. (Optional) Run the real FastAPI + Streamlit app

Launches the actual backend/frontend and gives you a clickable link to the
Streamlit chat UI, proxied through Colab.


Qdrant's local on-disk mode only allows one process to have `qdrant_data/`
open at a time. If you ran step 8 above, this notebook's own kernel is
still holding it open — the next cell releases that before starting the
FastAPI server as a separate process below.


In [ ]:
import app.tools as _tools_module

if _tools_module._qdrant_client is not None:
    _tools_module._qdrant_client.close()
    _tools_module._qdrant_client = None
    print("Released the in-notebook Qdrant client.")
else:
    print("No in-notebook Qdrant client was open — nothing to release.")


Released the in-notebook Qdrant client.


In [ ]:
import subprocess, time, requests

api_log = open("/content/uvicorn.log", "w")
api_proc = subprocess.Popen(
    ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=api_log, stderr=subprocess.STDOUT, cwd="/content/policy-agent",
)

for attempt in range(30):
    try:
        r = requests.get("http://localhost:8000/health", timeout=2)
        if r.status_code == 200:
            print("FastAPI server is up on port 8000.")
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError("FastAPI server did not start — check /content/uvicorn.log")


FastAPI server is up on port 8000.


In [ ]:
import subprocess, time, socket

st_log = open("/content/streamlit.log", "w")
st_proc = subprocess.Popen(
    [
        "streamlit", "run", "app/streamlit_app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
    ],
    stdout=st_log, stderr=subprocess.STDOUT, cwd="/content/policy-agent",
)

def port_open(port, host="localhost"):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(1)
        return s.connect_ex((host, port)) == 0

for attempt in range(30):
    if port_open(8501):
        print("Streamlit is up on port 8501.")
        break
    time.sleep(1)
else:
    raise RuntimeError("Streamlit did not start — check /content/streamlit.log")


Streamlit is up on port 8501.


In [ ]:
from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8501)")
print(f"Open your chat UI here: {url}")


Open your chat UI here: https://8501-m-s-kkb-ass1b2-u4g2wq8kxlpd-b.asia-southeast1-2.prod.colab.dev


To stop the background servers (e.g. before re-running earlier cells):
```python
api_proc.terminate()
st_proc.terminate()
ollama_proc.terminate()
```


## 10. (Optional) Persist your project + data to Google Drive

So you don't have to re-upload the zip and re-ingest every session.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import shutil
dest = "/content/drive/MyDrive/policy-agent-backup"
shutil.copytree("/content/policy-agent", dest, dirs_exist_ok=True)
print(f"Copied project (including qdrant_data/) to {dest}")


Mounted at /content/drive
Copied project (including qdrant_data/) to /content/drive/MyDrive/policy-agent-backup


In [ ]:
from app import agent

agent.SYSTEM_PROMPT = """You are an internal HR assistant for Acme Corp.

You have three tools:
- search_policies: for questions about company policy/rules (remote work, PTO, expenses, code of conduct, security).
- search_employee: for questions about a specific person (their manager, department, title, email).
- list_documents: to see what policy documents exist.

For every question about company policy, rules, procedures, or a specific
person, you MUST call the relevant tool before answering — even if you
think you already know the answer. Never answer a factual question from
memory alone. Do not skip the tool call because the question sounds
generic or you're unsure a document covers it — call search_policies (or
list_documents first, if unsure what's covered) and let the result decide.
The only messages you may answer without calling a tool are ones with no
factual content to look up, such as greetings or thanks.

If a tool returns nothing relevant, say plainly that you don't have that
information — do not guess or make something up. Keep answers concise and
cite which document a policy answer came from when relevant.
"""

ask("How many PTO days do I get per year?")
ask("What's the dog-walking policy?")

Q: How many PTO days do I get per year?
Tools used: []
A: The number of PTO (Paid Time Off) days you receive each year can vary depending on your position and the specific policies of Acme Corp. Typically, this information is outlined in the employee handbook or HR policy document. To find out exactly how many days you're entitled to, please refer to the Employee Handbook or contact Human Resources directly for clarification.

If you need assistance locating these documents, I can help guide you through that process.

Q: What's the dog-walking policy?
Tools used: []
A: I don't have information on specific policies like the dog-walking policy. Could you please provide more context or clarify your question? If it's about general company rules, I can check if there are any relevant documents that cover such topics.



{'answer': "I don't have information on specific policies like the dog-walking policy. Could you please provide more context or clarify your question? If it's about general company rules, I can check if there are any relevant documents that cover such topics.",
 'tool_calls': []}

In [ ]:
import subprocess
subprocess.run(["ollama", "pull", "qwen2.5:7b"], check=True)

from app import config
config.OLLAMA_CHAT_MODEL = "qwen2.5:7b"

ask("How many PTO days do I get per year?")
ask("What's the dog-walking policy?")


Q: How many PTO days do I get per year?
Tools used: ['search_policies']
A: According to the Paid Time Off (PTO) Policy document, full-time employees at Acme Corp accrue 2.083 PTO days per month, totaling 25 days per calendar year. This begins accruing from the first day of employment but cannot be used until after the 90-day probationary period ends. Employees may carry over a maximum of 5 unused PTO days into the following calendar year; any balance beyond that is forfeited on December 31st, unless local law requires otherwise.

For more detailed information or specific questions about your personal accrual, please consult with HR.

  [search_policies({'query': 'PTO days'}) ->]
  [source: pto_policy.pdf, score: 0.663]
Acme Corp  Paid Time Off (PTO) Policy
1. Accrual
----------
Full-time employees accrue 2.083 PTO days per month, totaling 25 days per calendar year. PTO
begins accruing from the first day of employment but cannot be used until after the 90-day
probationary peri...

Q: Wh

{'answer': 'The provided policies do not specifically mention a dog-walking policy. However, based on the documents searched:\n\n- For PTO requests, manager approval is required for requests of 1-2 days if submitted with 48 hours notice (PTO Policy).\n- The Code of Conduct does not include any specific rules regarding personal activities like dog walking.\n\nIf you have a specific concern or need to request time off for dog walking duties, it would be best to consult your manager and follow the PTO policy guidelines.',
 'tool_calls': [{'name': 'search_policies',
   'args': {'query': 'dog-walking policy'},
   'result': "[source: pto_policy.pdf, score: 0.486]\nmanager approval permitting.\n4. Sick Leave\n-------------\nSick leave is tracked separately from PTO and is not capped, subject to a medical certificate\nbeing provided for absences longer than 3 consecutive days.\n\n---\n\n[source: pto_policy.pdf, score: 0.478]\nAcme Corp  Paid Time Off (PTO) Policy\n1. Accrual\n----------\nFull-

Next session, instead of step 1's upload, you can restore straight from
Drive:
```python
from google.colab import drive
drive.mount("/content/drive")
import shutil, os
shutil.copytree("/content/drive/MyDrive/policy-agent-backup", "/content/policy-agent", dirs_exist_ok=True)
os.chdir("/content/policy-agent")
```
Then skip to step 2.


## Notes

- **Qdrant local mode** locks the `qdrant_data/` folder for exclusive
  access — if you re-run ingestion while `uvicorn` (step 9) is still
  running, stop it first (`api_proc.terminate()`).
- **You're done with Stage 0** when the smoke test in step 7 passes both
  cases and you can ask both kinds of questions (step 8, or the Streamlit
  UI in step 9) and get correct answers with no manual intervention. Stage
  1 of the plan wraps this same agent in a LangGraph self-healing loop —
  `app/tools.py` doesn't need to change.
